In [7]:
pip install python-dotenv langchain-openai langchain-chroma langchain-huggingface langchain-community langchain-text-splitters scikit-learn openai plotly numpy tiktoken sentence-transformers

   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/23.5 MB ? eta -:--:--
   - -------------------------------------- 0.8/23.5 MB 2.0 MB/s eta 0:00:12
   --- ------------------------------------ 1.8/23.5 MB 3.0 MB/s eta 0:00:08
   ------ --------------------------------- 3.9/23.5 MB 5.0 MB/s eta 0:00:04
   ------------ --------------------------- 7.1/23.5 MB 6.9 MB/s eta 0:00:03
   ------------------- -------------------- 11.3/23.5 MB 9.2 MB/s eta 0:00:02
   --------------------- ------------------ 12.8/23.5 MB 9.8 MB/s eta 0:00:02
   --------------------- ------------------ 12.8/23.5 MB 9.8 MB/s eta 0:00:02
   --------------------- ------------------ 12.8/23.5 MB 9.8 MB/s eta 0:00:02
   ------------------------------------ --- 21.2/23.5 MB 10.6 MB/s eta 0:00:01
   ------------------------------------- -- 22.0/23.5 MB 9.8 MB/s eta 0:00:01
   ----------

In [28]:
pip install gradio langchain-ollama


   -------------------- ------------------- 1/2 [langchain-ollama]
   ---------------------------------------- 2/2 [langchain-ollama]

Note: you may need to restart the kernel to use updated packages.


In [29]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
from openai import OpenAI
from langchain_ollama import ChatOllama

import plotly.graph_objects as go
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

In [9]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "llama3.2"
db_name = "vector_db"
load_dotenv(override=True)
client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [10]:
from pathlib import Path

knowledge = {}

kb_path = Path(
    r"D:\moved\projects\RAG_implementation\Code\telecom_demo_kb_large\documents"
)

# Get ONLY files, never folders
filenames = [
    p for p in kb_path.rglob("*")
    if p.is_file()
]

print(f"Found {len(filenames)} files")

# Show what was found
for p in filenames[:10]:
    print(p)

# Load only text files
for filename in filenames:
    

    name = filename.stem

    with filename.open("r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

print(f"Loaded {len(knowledge)} documents")


entire_knowledge_base = ""

for file_path in filenames:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 581 files
D:\moved\projects\RAG_implementation\Code\telecom_demo_kb_large\documents\alarm_guides\alarm_guide_001.md
D:\moved\projects\RAG_implementation\Code\telecom_demo_kb_large\documents\alarm_guides\alarm_guide_002.md
D:\moved\projects\RAG_implementation\Code\telecom_demo_kb_large\documents\alarm_guides\alarm_guide_003.md
D:\moved\projects\RAG_implementation\Code\telecom_demo_kb_large\documents\alarm_guides\alarm_guide_004.md
D:\moved\projects\RAG_implementation\Code\telecom_demo_kb_large\documents\alarm_guides\alarm_guide_005.md
D:\moved\projects\RAG_implementation\Code\telecom_demo_kb_large\documents\alarm_guides\alarm_guide_006.md
D:\moved\projects\RAG_implementation\Code\telecom_demo_kb_large\documents\alarm_guides\alarm_guide_007.md
D:\moved\projects\RAG_implementation\Code\telecom_demo_kb_large\documents\alarm_guides\alarm_guide_008.md
D:\moved\projects\RAG_implementation\Code\telecom_demo_kb_large\documents\alarm_guides\alarm_guide_009.md
D:\moved\projects\RAG_implemen

In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")  
tokens = tokenizer.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (146184 > 131072). Running this sequence through the model will result in indexing errors


Total tokens for llama3.2: 146,184


In [12]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("telecom_demo_kb_large/documents/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 580 documents


In [13]:
documents[1]


Document(metadata={'source': 'telecom_demo_kb_large\\documents\\alarm_guides\\alarm_guide_002.md', 'doc_type': 'alarm_guides'}, page_content='# Alarm Guide: BGP-NEIGHBOR-DOWN\n\n## Alarm meaning\n\nBGP-NEIGHBOR-DOWN is a synthetic major alarm associated with BGP peer. The alarm should be interpreted as an observation, not an automatic root-cause classification.\n\n## What commonly correlates\n\nRelated alarms may include LINK-DOWN, OPTICAL-RX-LOW, LATENCY-HIGH. Correlation across time and topology can reveal whether the alarm is primary or downstream.\n\n## False positives and secondary symptoms\n\nThe alarm may appear during planned maintenance, routing convergence, device restart, or another upstream fault. Check change windows and neighboring elements before escalating solely from alarm severity.\n\n## Recommended context\n\nAn RCA agent should retrieve the affected device, interface or peer, site, service dependencies, current telemetry, recent changes, and historical incidents mat

In [ ]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 2493 chunks
First chunk:

page_content='# Alarm Guide: BGP-NEIGHBOR-FLAP

## Alarm meaning

BGP-NEIGHBOR-FLAP is a synthetic major alarm associated with BGP peer. The alarm should be interpreted as an observation, not an automatic root-cause classification.

## What commonly correlates

Related alarms may include LINK-DOWN, HIGH-MEMORY, PACKET-LOSS. Correlation across time and topology can reveal whether the alarm is primary or downstream.

## False positives and secondary symptoms' metadata={'source': 'telecom_demo_kb_large\\documents\\alarm_guides\\alarm_guide_001.md', 'doc_type': 'alarm_guides'}


In [15]:
# Pick an embedding model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

d:\moved\projects\RAG_implementation\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Rahmeen\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2222.34it/s]


Vectorstore created with 2493 documents


In [16]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 2,493 vectors with 384 dimensions in the vector store


In [30]:
retriever = vectorstore.as_retriever()
llm = ChatOllama(model="llama3.2", temperature=0)

In [32]:
retriever.invoke("What is BGP-NEIGHBOR-FLAP?")

[Document(id='2d562708-7896-466d-a6e4-4e77d56a26cb', metadata={'source': 'telecom_demo_kb_large\\documents\\alarm_guides\\alarm_guide_001.md', 'doc_type': 'alarm_guides'}, page_content='# Alarm Guide: BGP-NEIGHBOR-FLAP\n\n## Alarm meaning\n\nBGP-NEIGHBOR-FLAP is a synthetic major alarm associated with BGP peer. The alarm should be interpreted as an observation, not an automatic root-cause classification.\n\n## What commonly correlates\n\nRelated alarms may include LINK-DOWN, HIGH-MEMORY, PACKET-LOSS. Correlation across time and topology can reveal whether the alarm is primary or downstream.\n\n## False positives and secondary symptoms'),
 Document(id='c823b681-f268-4992-8f8f-756db3328fa0', metadata={'source': 'telecom_demo_kb_large\\documents\\alarm_guides\\alarm_guide_017.md', 'doc_type': 'alarm_guides'}, page_content='# Alarm Guide: BGP-NEIGHBOR-FLAP\n\n## Alarm meaning\n\nBGP-NEIGHBOR-FLAP is a synthetic major alarm associated with BGP peer. The alarm should be interpreted as an obs

In [33]:
llm.invoke("What is BGP-NEIGHBOR-FLAP?")

AIMessage(content='BGP-NEIGHBOR-FLAP is a BGP (Border Gateway Protocol) event that indicates a flap in the BGP neighbor relationship. A flap occurs when a BGP neighbor\'s state changes, causing the BGP router to re-establish the neighbor relationship.\n\nWhen a BGP neighbor\'s state changes, the BGP router sends a BGP-NEIGHBOR-FLAP message to its peers, indicating the change in the neighbor\'s state. This message contains information about the neighbor\'s previous state, the reason for the change, and the new state of the neighbor.\n\nBGP-NEIGHBOR-FLAP messages are used to:\n\n1. Indicate a change in the BGP neighbor\'s state, such as a change from "up" to "down" or vice versa.\n2. Provide information about the reason for the change, such as a network failure or a change in the neighbor\'s configuration.\n3. Allow the BGP router to re-establish the neighbor relationship and update its routing table accordingly.\n\nBGP-NEIGHBOR-FLAP messages are an important part of BGP\'s flap detectio

In [43]:
SYSTEM_PROMPT_TEMPLATE = """You are an AI-assisted Root Cause Analysis (RCA) Agent for a telecom Network Operations Center (NOC).

Your primary responsibility is to analyze network and IT incidents using the provided knowledge base, incident information, 
alarms, telemetry, logs, topology context, historical incidents, postmortems, runbooks, known errors, vendor troubleshooting guides,
 change records, and performance baselines. Politely decline irrelevant question and do not answer it. IF there is not relevant context in the knowladge base. say so instead of making assumptions
 Relevant context:
 {context}
 """

In [44]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    print(docs)
    return response.content

In [ ]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


[Document(id='ad07e18d-8345-4388-9bef-2ff447df29f8', metadata={'doc_type': 'historical_incidents', 'source': 'telecom_demo_kb_large\\documents\\historical_incidents\\incident_0078.md'}, page_content='# Historical Incident INC-2025-0078\n\n## Incident summary\n\nINC-2025-0078 affected RR-PES-065 at POP-PES-01. The primary symptom was Interface errors: CRC/input errors increase materially. The incident was classified as P1 and lasted approximately 23 minutes.\n\n## Observed evidence'), Document(id='62b56471-9751-4f3b-a522-07a4ce945918', metadata={'source': 'telecom_demo_kb_large\\documents\\historical_incidents\\incident_0103.md', 'doc_type': 'historical_incidents'}, page_content='# Historical Incident INC-2026-0103\n\n## Incident summary\n\nINC-2026-0103 affected RR-PES-065 at POP-PES-01. The primary symptom was Link down: interface operational state is down. The incident was classified as P2 and lasted approximately 83 minutes.\n\n## Observed evidence'), Document(id='b2c45af6-6e42-4248

In [48]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange', 'purple','black','lilac', 'pink','yellow','brown'][['alarm_guides', 'change_context', 'historical_incidents', 'known_errors', 'performance_baselines', 'postmortems','runbooks','site_operations','topology_context','vendor_troubleshooting'].index(t)] for t in doc_types]

In [53]:
from sklearn.manifold import TSNE
import plotly.graph_objects as go

tsne = TSNE(
    n_components=2,
    perplexity=min(30, len(vectors) - 1),
    random_state=42
)

reduced_vectors = tsne.fit_transform(vectors)

fig = go.Figure(
    data=[
        go.Scatter(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            mode="markers",
            marker=dict(
                size=5,
                opacity=0.8
            ),
            text=[
                f"Type: {t}<br>Text: {d[:100]}..."
                for t, d in zip(doc_types, documents)
            ],
            hoverinfo="text"
        )
    ]
)

fig.update_layout(
    title="2D Chroma Vector Store Visualization",
    xaxis_title="x",
    yaxis_title="y",
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show(renderer="browser")

In [55]:
from sklearn.manifold import TSNE
import plotly.graph_objects as go

# 3D t-SNE
tsne = TSNE(
    n_components=3,
    perplexity=min(30, len(vectors) - 1),
    random_state=42
)

reduced_vectors = tsne.fit_transform(vectors)


# Give each document type a different color
type_colors = {
    "historical_incidents": "red",
    "postmortems": "blue",
    "runbooks": "green",
    "known_errors": "orange",
    "vendor_troubleshooting": "purple",
    "alarm_guides": "brown",
    "change_context": "pink",
    "performance_baselines": "cyan",
    "site_operations": "gold",
    "topology_context": "black"
}

# Create a color for every document
point_colors = [
    type_colors.get(t, "gray")
    for t in doc_types
]


# Create the 3D scatter plot
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=reduced_vectors[:, 0],
            y=reduced_vectors[:, 1],
            z=reduced_vectors[:, 2],

            mode="markers",

            marker=dict(
                size=5,
                color=point_colors,
                opacity=0.8
            ),

            text=[
                f"Type: {t}<br>Text: {d[:100]}..."
                for t, d in zip(doc_types, documents)
            ],

            hoverinfo="text"
        )
    ]
)


fig.update_layout(
    title="3D Chroma Vector Store Visualization",

    scene=dict(
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="z"
    ),

    width=900,
    height=700,

    margin=dict(
        r=10,
        b=10,
        l=10,
        t=40
    )
)

fig.show(renderer="browser")